# Lab 4 — Notebook 03: Model Comparison, Cross-Validation, and Final Evaluation

**Duration:** 55 minutes (core route)  
**Dataset:** `data/digits.csv`, unchanged from Notebook 02.  
**Prerequisites:** Classification, pipelines, validation metrics, k-NN, trees, logistic regression, and random forests.

## Learning objectives

- Observe how model complexity changes training and validation performance.
- Shortlist configurations using validation results.
- Compare two full pipelines with five-fold stratified cross-validation.
- Interpret mean accuracy and fold-to-fold variation.
- Refit the selected configuration and evaluate once on the untouched test set.

**Workflow:** small validation experiments → shortlist two → cross-validation on development data → select → refit → final test.

Each notebook runs independently. The split is identical to Notebook 02. All guided examples work without TODO answers. Section budgets include exercises: 5 + 8 + 8 + 7 + 17 + 7 + 3 = **55 minutes**. Optional extensions are outside that budget.


## 1. Load the digit dataset and create three splits (5 min, including Exercise 1)

The dataset contains 8 × 8 grayscale handwritten digits, represented as 64 numerical pixel values (`pixel_0` to `pixel_63`). The column `target` contains the correct digit from 0 to 9.

We use three subsets:

- **Training set:** fit each model and learn patterns.
- **Validation set:** compare candidate models and choose hyperparameters.
- **Test set:** estimate performance on unseen data **only after selection**.

We first reserve 20% for testing. We then reserve 20% of the *remaining* observations for validation, resulting in approximately **64% training / 16% validation / 20% testing**. `stratify` preserves the proportions of digit labels in each split. **These are the same splits used in Notebook 02**, even if you run this notebook independently.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

csv_path = Path("data/digits.csv")
if not csv_path.is_file():
    raise FileNotFoundError(
        "Cannot find data/digits.csv. Open JupyterLab in the lab04 directory "
        "or set its working directory to lab04."
    )

df = pd.read_csv(csv_path)
pixel_columns = [f"pixel_{i}" for i in range(64)]
X = df[pixel_columns]
y = df["target"]

X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_development, y_development,
    test_size=0.20, random_state=42, stratify=y_development
)

print("Training:", X_train.shape, "(model fitting)")
print("Validation:", X_val.shape, "(model selection)")
print("Test:", X_test.shape, "(final evaluation only)")
print("Classes:", sorted(y.unique()))
assert len(X_train) + len(X_val) + len(X_test) == len(df)
assert X_train.index.intersection(X_val.index).empty
assert X_development.index.intersection(X_test.index).empty


### Core exercise 1 — Explain the split

Calculate the training and validation percentages. Why is the test set excluded from **all** model selection? No need to examine its labels or images.

**Your explanation:** TODO.


In [ ]:
# TODO: calculate train and validation percentages
# TODO: explain why the test set is reserved


## 2. k-Nearest neighbors: Underfitting and overfitting (8 min, including Exercise 2)

A **hyperparameter** is a model setting that we choose *before fitting*. For k-nearest neighbors (k-NN), the hyperparameter `n_neighbors` controls how many neighbors vote on a prediction.

- With `k=1`, the model can effectively memorize its training observations. This often yields very high training accuracy, but may not generalize as well.
- A larger `k` produces a smoother, simpler decision rule. If `k` becomes too large, the classifier may **underfit**, ignoring useful local patterns.

We will train several values of `k` and plot **training accuracy** and **validation accuracy**. Scaling is fitted on training data only by placing `StandardScaler` inside a `Pipeline`.

> **Watch for:** A large gap between training and validation accuracy can suggest overfitting. A model doing poorly on both can suggest underfitting.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

k_values = [1, 5, 15, 41]
knn_train_scores = []
knn_val_scores = []

for k in k_values:
    model = Pipeline([
        ("scale", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_train, y_train)
    knn_train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    knn_val_scores.append(accuracy_score(y_val, model.predict(X_val)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, knn_train_scores, "o-", label="Training accuracy")
ax.plot(k_values, knn_val_scores, "s-", label="Validation accuracy")
ax.set(xlabel="Number of neighbors (k)", ylabel="Accuracy",
       title="k-NN: training vs validation accuracy")
ax.set_xticks(k_values)
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

best_k = k_values[int(np.argmax(knn_val_scores))]
print(f"Best k by validation accuracy: {best_k}")
print(f"Best validation accuracy: {max(knn_val_scores):.3f}")


### Exercise 2 — Interpret the k-NN learning curves

1. Compare training and validation accuracy at `k=1` and `k=41`.
2. Which value of `k` would you choose using **validation data only**? Is it necessarily `k=1`?
3. Find the training–validation accuracy gap for `k=1` and for `k=15`.

**Your observations:** *TODO: Describe where you see possible overfitting or underfitting and explain why.*


In [ ]:
# TODO: compute and print train-validation accuracy gaps for k=1 and k=15
# Hint: k_values.index(1) gives the position of k=1


## 3. Decision trees: Investigating model complexity (8 min, including Exercise 3)

A decision tree repeatedly splits the feature space using questions about pixel values. Its `max_depth` controls how many successive questions the tree can ask.

- **Shallow tree:** too few decisions can miss important patterns (**underfitting**).
- **Deep tree:** may learn very specific details of training examples (**overfitting**).

Unlike k-NN, decision trees do not require feature scaling for these experiments.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = [1, 4, 10, None]
tree_train_scores = []
tree_val_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    tree_train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    tree_val_scores.append(accuracy_score(y_val, tree.predict(X_val)))

labels = [str(d) if d is not None else "None" for d in depths]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(len(depths)), tree_train_scores, "o-", label="Training accuracy")
ax.plot(range(len(depths)), tree_val_scores, "s-", label="Validation accuracy")
ax.set(xlabel="Maximum tree depth", ylabel="Accuracy",
       title="Decision tree: training vs validation accuracy")
ax.set_xticks(range(len(depths)), labels)
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

best_depth = depths[int(np.argmax(tree_val_scores))]
print(f"Best tree depth by validation accuracy: {best_depth}")
print(f"Best tree validation accuracy: {max(tree_val_scores):.3f}")


### Exercise 3 — Investigate tree complexity

1. Compare the scores for a very shallow tree (`max_depth=1`) and the unrestricted tree (`max_depth=None`).
2. Find the depth with the **largest** training–validation accuracy gap.
3. Explain why achieving 100% training accuracy does **not** guarantee a perfect classifier.

**Your explanation:** *TODO: Explain the relationship between tree depth, overfitting, and generalization.*

In [ ]:
# TODO: calculate tree_train_scores - tree_val_scores
# TODO: print the depth with the largest gap

## 4. Shortlist two configurations (7 min, including reflection)

**Core.** Bring the four families together: best validation k-NN, best validation tree, and the fixed logistic-regression and random-forest configurations from Notebook 02. Recreate them here so this notebook works independently (using sklearn.base.clone, which creates a fresh, unfitted copy of an estimator).

Select the **two highest validation accuracies**. For exact ties, sort alphabetically by model name. Freeze this shortlist before cross-validation. This small exercise illustrates selection, not an exhaustive search.

Every candidate is a `Pipeline`. Trees use `passthrough` preprocessing (— a placeholder that passes the data through unchanged, with no transformation applied); the other models learn a scaler inside their pipeline.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

candidates = {
    "k-NN": Pipeline([
        ("preprocess", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=best_k)),
    ]),
    "Decision tree": Pipeline([
        ("preprocess", "passthrough"),
        ("classifier", DecisionTreeClassifier(max_depth=best_depth, random_state=42)),
    ]),
    "Logistic regression": Pipeline([
        ("preprocess", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    "Random forest": Pipeline([
        ("preprocess", "passthrough"),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)),
    ]),
}
validation_rows = []
for name, candidate in candidates.items():
    fitted = clone(candidate).fit(X_train, y_train)
    validation_rows.append({
        "model": name,
        "train_accuracy": accuracy_score(y_train, fitted.predict(X_train)),
        "validation_accuracy": accuracy_score(y_val, fitted.predict(X_val)),
    })
comparison = pd.DataFrame(validation_rows).sort_values(
    ["validation_accuracy", "model"], ascending=[False, True]
).reset_index(drop=True)
display(comparison.round(4))
shortlist = comparison["model"].head(2).tolist()
print("Frozen shortlist:", shortlist)


### Core reflection — What was selected?

Name the two candidates. Did the forest beat the single tree on validation data? Why might the ranking depend on this particular split?

**Your explanation:** TODO.


In [ ]:
# TODO: report the shortlisted candidates and their validation accuracies


## 5. Cross-validation of the full pipelines (17 min, including exercise)

**Core.** A single validation split gives one estimate. Five-fold cross-validation divides the **development set** into five parts, fits on four, and scores on the remaining part. Repeat so each part is scored once. Stratification approximately preserves class proportions.

We now reuse training + validation as development data. The earlier validation set is no longer a separate holdout. The final test set remains untouched.

Pass the **unfitted full pipeline** to `cross_val_score`, not an already scaled feature matrix. Each fold gets a fresh estimator; any scaler learns only from that fold's training rows. Trees need no scaler but use the same folds for fairness.

Report mean accuracy and standard deviation across folds (the code uses ddof=0, which tells NumPy to compute the population, not sample, standard deviation). The standard deviation describes variability across these folds; it is **not a confidence interval**, and folds are not independent experiments.

**Selection limitation:** The shortlist was chosen using these development observations. These CV scores therefore support model selection; they are not an unbiased assessment of the entire selection procedure. We reserve that role for the independent final test. Nested CV is beyond this lab.

**Rule fixed in advance:** highest mean CV accuracy wins; exact ties are resolved alphabetically. A small difference is not proof of a practically important advantage.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Reuse exactly the same fold indices for both candidates.
folds = list(cv.split(X_development, y_development))
cv_scores = {}
for name in shortlist:
    cv_scores[name] = cross_val_score(
        clone(candidates[name]), X_development, y_development,
        cv=folds, scoring="accuracy", n_jobs=1, error_score="raise",
    )
fold_table = pd.DataFrame(cv_scores, index=[f"Fold {i}" for i in range(1, 6)])
display(fold_table.round(4))
cv_summary = pd.DataFrame([
    {"model": name, "mean_accuracy": scores.mean(),
     "std_accuracy": scores.std(ddof=0)}
    for name, scores in cv_scores.items()
]).sort_values(["mean_accuracy", "model"], ascending=[False, True]).reset_index(drop=True)
display(cv_summary.round(4))

ax = fold_table.plot(marker="o", figsize=(8, 4))
ax.set(xlabel="Held-out development fold", ylabel="Accuracy",
       title="Same folds, two candidate pipelines", ylim=(0, 1.02))
plt.tight_layout()
plt.show()

winner_name = cv_summary.loc[0, "model"]
selected_model = clone(candidates[winner_name])
print("Selected BEFORE inspecting test results:", winner_name)

### Core exercise — Interpret the evidence

1. Calculate the difference between the two mean accuracies in **percentage points**.
2. Does the same candidate win on every fold?
3. Why is `StandardScaler().fit_transform(X_development)` before cross-validation incorrect?
4. Why do we still need the final test set?

**Your interpretation:** TODO — Refer to the means and variability without claiming statistical significance.

In [ ]:
# TODO: calculate the mean-score gap in percentage points
# TODO: compare the candidate scores fold by fold

## 6. Refit and evaluate once (7 min, including reflection)

The configuration is now fixed using **development results only**. Fit a fresh selected pipeline on all development observations, then evaluate the reserved test set once. Do not return to model selection after viewing the result.

Report accuracy, the number of correct predictions, and a confusion matrix. This is an estimate on this test sample, not a guarantee for all future images. The final score can differ from CV because training-set sizes and evaluation observations differ.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

selected_model.fit(X_development, y_development)
test_predictions = selected_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final selected model: {winner_name}")
print(f"Held-out test accuracy: {test_accuracy:.3f}")
print(f"Correct predictions: {(test_predictions == y_test.to_numpy()).sum()} / {len(y_test)}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=list(range(10)),
    cmap="Blues", values_format="d", ax=ax
)
ax.set_title(f"Final test confusion matrix — {winner_name}")
plt.show()

### Core reflection — State the result

Write the selected model, its mean CV accuracy, and its final test accuracy. Explain why a different test score does not authorize tuning against that test set.

### Extension — Inspect one final test error (optional, 3 min)

Find and display a misclassified test image, if any. Handle the no-errors case. Explain the error without modifying the model.


In [ ]:
# TODO: locate a test example for which test_predictions != y_test
# TODO: display the 8x8 pixel image with plt.imshow(..., cmap="gray_r")
# TODO: print the true label and predicted label

## 7. Core wrap-up (3 min)

- What distinguishes a parameter from a hyperparameter?
- What does each CV fold learn, and what does it only evaluate?
- Why is the CV standard deviation not a confidence interval?
- Why is the final test separate from both shortlisting and CV?

**Deliverable:** Complexity plots, a shortlist, fold scores, a justified selection, and one final test result. Restart and Run All to check reproducibility; do not use repeated test results for further tuning.

**Next: Lab 5.** Apply the same evaluation discipline to neural networks and webcam projects.


## Team checkpoint — From digits to your webcam project

Use the shared lab discussion time (outside the 55-minute notebook budget). Record:

- **Task:** What should the camera system predict?
- **Baseline:** What simple or pretrained approach will you compare against?
- **Evaluation:** What measurement would show success? Object detection will also require checking box localization in Lab 5.
- **Independent data:** How will you collect validation and final test examples separately from training? Keep frames from the same short recording in one partition.
- **Failure case:** Which lighting, background, or viewing condition might cause trouble?

**Team response:** TODO — Write a short plan, not just a model name.